In [1]:
import pandas as pd
import plotly.express as px

In [2]:
train_panel_in = "../../../data/final/train_panel.parquet"
bike_panel_in = "../../../data/final/bike_panel.parquet"
uber_panel_in = "../../../data/final/uber_panel.parquet"

parallel_out = "../../../reports/replication/parallel_trends.jpeg"

In [3]:
panel = pd.concat([pd.read_parquet(train_panel_in),
                   pd.read_parquet(bike_panel_in),
                   pd.read_parquet(uber_panel_in)],
                  keys=['train','bike','uber'], names=['transit']).reset_index('transit')


## Parallel Trends

PT is actually pretty convincing.

In [4]:
plot_data = panel.assign(nearby = panel['UCMP'].map({0:False, 1:True}))
plot_data = plot_data.groupby(['date','nearby','transit'])['rides'].sum().reset_index()
means = plot_data.groupby(['nearby','transit'])['rides'].mean().rename('mean').reset_index()
first = plot_data.sort_values('date')[['nearby','transit','rides']].iloc[:6].rename(columns={'rides':'first'})
plot_data = plot_data.merge(means, on=['nearby','transit']).merge(first, on=['nearby','transit'])
plot_data = plot_data.assign(demean = 100 *   plot_data['rides'] / plot_data['first'])

In [5]:
chart = px.line(plot_data, x='date', y='demean', 
        line_dash='nearby', facet_row='transit',
        color_discrete_sequence=['black'],
        title= 'Rides (normalized to starting date)',
        labels={'date':'', 'demean':'',
                'nearby':'Near DNC'},
        template='simple_white')
chart.add_vrect(x0="2024-08-19", x1="2024-08-22", 
                fillcolor="gray", opacity=.5)
chart.write_image(parallel_out)
!cp {parallel_out} ../../../../eric-mc2-cv/static/img
chart.show()
